# Regression Problem of the given datasets RG-Credit & RG-Wage 
### the Decision Tree Regressor vs. XGBoost Regressor comparison ###



## Import Libraries

In [3]:
# %pip install xgboost numpy pandas matplotlib scikit-learn
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.tree import DecisionTreeRegressor, export_text
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from xgboost import XGBRegressor

SEED = 445

## Making our Custom MSE Metric
without using any library function we make our own MSE caluculation function

In [4]:
def custom_mse(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=np.float64)
    y_pred = np.asarray(y_pred, dtype=np.float64)
    n = y_true.shape[0]
    residuals = y_true - y_pred
    squared_residuals = residuals ** 2
    sum_squared = np.sum(squared_residuals)  
    mse = sum_squared / n                   
    return float(mse)

---
## Dataset 1: Credit Dataset (label: `Balance`)

### Loading Dataset & Splitting
we perform a sequential split on the first 70% as train, next 15% as validation and last 15% as test 


In [5]:
SEED = 445
df = pd.read_csv("../datasets/RG-Credit.csv")
df = df.dropna()
print(df.shape)

X = df.drop(columns=["Balance"])
y = df["Balance"]

# Fix: shuffle=False for a sequential (non-shuffled) split
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.30, random_state=SEED, shuffle=False)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.50, random_state=SEED, shuffle=False)

print("Train size:", X_train.shape[0])
print("Validation size:", X_val.shape[0])
print("Test size:", X_test.shape[0])

(400, 11)
Train size: 280
Validation size: 60
Test size: 60


### Feature Encoding
Using one hot encoder for only the categorical columns (`Own`, `Student`, `Married`, `Region`)
the numeric columns (`Income`, `Limit`, `Rating`, `Cards`,`Age`, `Education`) are kept as continuous features rather than being passed into the encoder.

In [6]:
cat_cols = X_train.select_dtypes(include=["object", "string"]).columns.tolist()
num_cols = [c for c in X_train.columns if c not in cat_cols]
print("Categorical columns:", cat_cols)
print("Numeric columns:", num_cols)

encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False, drop="first")

X_train_cat = encoder.fit_transform(X_train[cat_cols])
X_val_cat = encoder.transform(X_val[cat_cols])
X_test_cat = encoder.transform(X_test[cat_cols])

X_train_encoded = np.hstack([X_train[num_cols].values, X_train_cat])
X_val_encoded = np.hstack([X_val[num_cols].values, X_val_cat])
X_test_encoded = np.hstack([X_test[num_cols].values, X_test_cat])

print("Encoded feature matrix shape (train):", X_train_encoded.shape)

Categorical columns: ['Own', 'Student', 'Married', 'Region']
Numeric columns: ['Income', 'Limit', 'Rating', 'Cards', 'Age', 'Education']
Encoded feature matrix shape (train): (280, 11)


### Decision Tree Regressor

In [7]:
# Decision tree hyperparameter
DT_Hyperparameters = {
    "max_depth": [2, 3, 4, 5, 6, 7, 8, 9, 10],
    "min_samples_split": [2, 5, 10, 20],
    "min_samples_leaf": [1, 2, 4, 8],
    "criterion": ["squared_error", "absolute_error"]
}
results = []
best_mse = 1e10  # theoretically an infinite number for comparison
best_performing_model = None
best_hyperparameters = None
for depth in DT_Hyperparameters["max_depth"]:
    for min_samples_split in DT_Hyperparameters["min_samples_split"]:
        for min_samples_leaf in DT_Hyperparameters["min_samples_leaf"]:
            for criterion in DT_Hyperparameters["criterion"]:
                model = DecisionTreeRegressor(max_depth=depth,
                                              min_samples_split=min_samples_split,
                                              min_samples_leaf=min_samples_leaf,
                                              criterion=criterion, random_state=SEED)
                model.fit(X_train_encoded, y_train)
                y_pred = model.predict(X_val_encoded)
                mse = custom_mse(y_val, y_pred)
                results.append((depth, min_samples_split, min_samples_leaf, criterion, mse))
                if mse < best_mse:
                    best_mse = mse
                    best_performing_model = model
                    best_hyperparameters = (depth, min_samples_split, min_samples_leaf, criterion)

results_df = (pd.DataFrame(results, columns=["max_depth", "min_samples_split", "min_samples_leaf", "criterion", "mse"])
              .sort_values(by="mse")
              .reset_index(drop=True)
              )
print("Best Hyperparameters:", best_hyperparameters)
print("Best Validation MSE:", best_mse)
print("Model:", best_performing_model)

Best Hyperparameters: (7, 2, 1, 'absolute_error')
Best Validation MSE: 13476.016666666666
Model: DecisionTreeRegressor(criterion='absolute_error', max_depth=7, random_state=445)


### Decision Tree — Final Test Set Evaluation

In [8]:
dt_credit_model = best_performing_model
dt_credit_val_mse = best_mse
dt_credit_test_pred = dt_credit_model.predict(X_test_encoded)
dt_credit_test_mse = custom_mse(y_test, dt_credit_test_pred)

print("Decision Tree (Credit) — Best Hyperparameters:", best_hyperparameters)
print("Decision Tree (Credit) — Validation MSE:", dt_credit_val_mse)
print("Decision Tree (Credit) — Test MSE:", dt_credit_test_mse)

Decision Tree (Credit) — Best Hyperparameters: (7, 2, 1, 'absolute_error')
Decision Tree (Credit) — Validation MSE: 13476.016666666666
Decision Tree (Credit) — Test MSE: 24064.7375


### XGBoost Regressor


In [9]:
XGB_GRID = {
    "max_depth": [3, 5, 7],
    "learning_rate": [0.01, 0.05, 0.1],
    "n_estimators": [100, 300],
    "subsample": [0.8, 1.0],
    "colsample_bytree": [0.8, 1.0],
    "min_child_weight": [1, 5],
    "gamma": [0, 0.1],
}
results = []

best_mse = float("inf")
best_model = None
best_params = None

rng = np.random.RandomState(SEED)

for _ in range(60):
    max_depth = rng.choice(XGB_GRID["max_depth"])
    learning_rate = rng.choice(XGB_GRID["learning_rate"])
    n_estimators = rng.choice(XGB_GRID["n_estimators"])
    subsample = rng.choice(XGB_GRID["subsample"])
    colsample_bytree = rng.choice(XGB_GRID["colsample_bytree"])
    min_child_weight = rng.choice(XGB_GRID["min_child_weight"])
    gamma = rng.choice(XGB_GRID["gamma"])

    model = XGBRegressor(
        objective="reg:squarederror",
        random_state=SEED,
        n_jobs=4,
        verbosity=0,
        max_depth=max_depth,
        learning_rate=learning_rate,
        n_estimators=n_estimators,
        subsample=subsample,
        colsample_bytree=colsample_bytree,
        min_child_weight=min_child_weight,
        gamma=gamma
    )

    model.fit(X_train_encoded, y_train)
    predictions = model.predict(X_val_encoded)
    mse = custom_mse(y_val, predictions)

    results.append({
        "max_depth": max_depth,
        "learning_rate": learning_rate,
        "n_estimators": n_estimators,
        "subsample": subsample,
        "colsample_bytree": colsample_bytree,
        "min_child_weight": min_child_weight,
        "gamma": gamma,
        "validation_mse": mse
    })
    if mse < best_mse:
        best_mse = mse
        model2 = model
        best_params = {
            "max_depth": max_depth,
            "learning_rate": learning_rate,
            "n_estimators": n_estimators,
            "subsample": subsample,
            "colsample_bytree": colsample_bytree,
            "min_child_weight": min_child_weight,
            "gamma": gamma
        }

results_df = pd.DataFrame(results).sort_values("validation_mse").reset_index(drop=True)

print("Best Hyperparameters:", best_params)
print("Best Validation MSE:", best_mse)
print("Model:", model2)

Best Hyperparameters: {'max_depth': np.int64(3), 'learning_rate': np.float64(0.1), 'n_estimators': np.int64(300), 'subsample': np.float64(1.0), 'colsample_bytree': np.float64(0.8), 'min_child_weight': np.int64(1), 'gamma': np.float64(0.1)}
Best Validation MSE: 7060.3042045696375
Model: XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=np.float64(0.8), device=None,
             early_stopping_rounds=None, enable_categorical=True,
             eval_metric=None, feature_types=None, feature_weights=None,
             gamma=np.float64(0.1), grow_policy=None, importance_type=None,
             interaction_constraints=None, learning_rate=np.float64(0.1),
             max_bin=None, max_cat_threshold=None, max_cat_to_onehot=None,
             max_delta_step=None, max_depth=np.int64(3), max_leaves=None,
             min_child_weight=np.int64(1), missing=nan,
             monotone_constraints=None,

### XGBoost — Final Test Set Evaluation

In [10]:
xgb_credit_model = model2
xgb_credit_val_mse = best_mse
xgb_credit_test_pred = xgb_credit_model.predict(X_test_encoded)
xgb_credit_test_mse = custom_mse(y_test, xgb_credit_test_pred)

print("XGBoost (Credit) — Best Hyperparameters:", best_params)
print("XGBoost (Credit) — Validation MSE:", xgb_credit_val_mse)
print("XGBoost (Credit) — Test MSE:", xgb_credit_test_mse)

XGBoost (Credit) — Best Hyperparameters: {'max_depth': np.int64(3), 'learning_rate': np.float64(0.1), 'n_estimators': np.int64(300), 'subsample': np.float64(1.0), 'colsample_bytree': np.float64(0.8), 'min_child_weight': np.int64(1), 'gamma': np.float64(0.1)}
XGBoost (Credit) — Validation MSE: 7060.3042045696375
XGBoost (Credit) — Test MSE: 5181.186680101869


---
## Dataset 2: Wage Dataset (target: `wage`)

### Loading Dataset & Sequential Train / Validation / Test Splitting

In [11]:
SEED = 445
df = pd.read_csv("../datasets/RG-Wage.csv")
df = df.dropna()
df = df.drop(columns=["logwage", "region"]) #Because they can cause data leak 
print(df.shape)

X = df.drop(columns=["wage"])
y = df["wage"]


X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.30, random_state=SEED, shuffle=False)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.50, random_state=SEED, shuffle=False)

print("Train size:", X_train.shape[0])
print("Validation size:", X_val.shape[0])
print("Test size:", X_test.shape[0])

(3000, 9)
Train size: 2100
Validation size: 450
Test size: 450


### Feature Encoding
only the categorical columns (`maritl`, `race`, `education`, `jobclass`,
`health`, `health_ins`) are one-hot encoded; `year` and `age` are kept as
numeric features instead of being passed into the encoder.

In [12]:
cat_cols = X_train.select_dtypes(include=["object", "string"]).columns.tolist()
num_cols = [c for c in X_train.columns if c not in cat_cols]
print("Categorical columns:", cat_cols)
print("Numeric columns:", num_cols)

encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False, drop="first")

X_train_cat = encoder.fit_transform(X_train[cat_cols])
X_val_cat = encoder.transform(X_val[cat_cols])
X_test_cat = encoder.transform(X_test[cat_cols])

X_train_encoded = np.hstack([X_train[num_cols].values, X_train_cat])
X_val_encoded = np.hstack([X_val[num_cols].values, X_val_cat])
X_test_encoded = np.hstack([X_test[num_cols].values, X_test_cat])

print("Encoded feature matrix shape (train):", X_train_encoded.shape)

Categorical columns: ['maritl', 'race', 'education', 'jobclass', 'health', 'health_ins']
Numeric columns: ['year', 'age']
Encoded feature matrix shape (train): (2100, 16)


### Decision Tree Regressor

In [13]:
# Descion tree
DT_Hyperparameters = {
    "max_depth": [2, 3, 4, 5, 6, 7, 8, 9, 10],
    "min_samples_split": [2, 5, 10, 20],
    "min_samples_leaf": [1, 2, 4, 8],
    "criterion": ["squared_error", "absolute_error"]
}
results = []
best_mse = 1e10  # theoriticially an infinite number for comaparisoon
best_performing_model = None
best_hyperparameters = None
for depth in DT_Hyperparameters["max_depth"]:
    for min_samples_split in DT_Hyperparameters["min_samples_split"]:
        for min_samples_leaf in DT_Hyperparameters["min_samples_leaf"]:
            for criterion in DT_Hyperparameters["criterion"]:
                model = DecisionTreeRegressor(max_depth=depth,
                                              min_samples_split=min_samples_split,
                                              min_samples_leaf=min_samples_leaf,
                                              criterion=criterion, random_state=SEED)
                model.fit(X_train_encoded, y_train)
                y_pred = model.predict(X_val_encoded)
                mse = custom_mse(y_val, y_pred)
                results.append((depth, min_samples_split, min_samples_leaf, criterion, mse))
                if mse < best_mse:
                    best_mse = mse
                    best_performing_model = model
                    best_hyperparameters = (depth, min_samples_split, min_samples_leaf, criterion)

results_df = (pd.DataFrame(results, columns=["max_depth", "min_samples_split", "min_samples_leaf", "criterion", "mse"])
              .sort_values(by="mse")
              .reset_index(drop=True)
              )
print("Best Hyperparameters:", best_hyperparameters)
print("Best Validation MSE:", best_mse)
print("Model:", best_performing_model)

Best Hyperparameters: (6, 5, 2, 'squared_error')
Best Validation MSE: 1303.5211076279738
Model: DecisionTreeRegressor(max_depth=6, min_samples_leaf=2, min_samples_split=5,
                      random_state=445)


### Decision Tree — Final Test Set Evaluation

In [14]:
dt_wage_model = best_performing_model
dt_wage_val_mse = best_mse
dt_wage_test_pred = dt_wage_model.predict(X_test_encoded)
dt_wage_test_mse = custom_mse(y_test, dt_wage_test_pred)

print("Decision Tree (Wage) — Best Hyperparameters:", best_hyperparameters)
print("Decision Tree (Wage) — Validation MSE:", dt_wage_val_mse)
print("Decision Tree (Wage) — Test MSE:", dt_wage_test_mse)

Decision Tree (Wage) — Best Hyperparameters: (6, 5, 2, 'squared_error')
Decision Tree (Wage) — Validation MSE: 1303.5211076279738
Decision Tree (Wage) — Test MSE: 1298.7330209484035


### XGBoost Regressor


In [15]:
XGB_GRID = {
    "max_depth": [3, 5, 7],
    "learning_rate": [0.01, 0.05, 0.1],
    "n_estimators": [100, 300],
    "subsample": [0.8, 1.0],
    "colsample_bytree": [0.8, 1.0],
    "min_child_weight": [1, 5],
    "gamma": [0, 0.1],
}
results = []

best_mse = float("inf")
best_model = None
best_params = None

rng = np.random.RandomState(SEED)

for _ in range(60):
    max_depth = rng.choice(XGB_GRID["max_depth"])
    learning_rate = rng.choice(XGB_GRID["learning_rate"])
    n_estimators = rng.choice(XGB_GRID["n_estimators"])
    subsample = rng.choice(XGB_GRID["subsample"])
    colsample_bytree = rng.choice(XGB_GRID["colsample_bytree"])
    min_child_weight = rng.choice(XGB_GRID["min_child_weight"])
    gamma = rng.choice(XGB_GRID["gamma"])

    model = XGBRegressor(
        objective="reg:squarederror",
        random_state=SEED,
        n_jobs=4,
        verbosity=0,
        max_depth=max_depth,
        learning_rate=learning_rate,
        n_estimators=n_estimators,
        subsample=subsample,
        colsample_bytree=colsample_bytree,
        min_child_weight=min_child_weight,
        gamma=gamma
    )

    model.fit(X_train_encoded, y_train)
    predictions = model.predict(X_val_encoded)
    mse = custom_mse(y_val, predictions)

    results.append({
        "max_depth": max_depth,
        "learning_rate": learning_rate,
        "n_estimators": n_estimators,
        "subsample": subsample,
        "colsample_bytree": colsample_bytree,
        "min_child_weight": min_child_weight,
        "gamma": gamma,
        "validation_mse": mse
    })
    if mse < best_mse:
        best_mse = mse
        model2 = model
        best_params = {
            "max_depth": max_depth,
            "learning_rate": learning_rate,
            "n_estimators": n_estimators,
            "subsample": subsample,
            "colsample_bytree": colsample_bytree,
            "min_child_weight": min_child_weight,
            "gamma": gamma
        }

results_df = pd.DataFrame(results).sort_values("validation_mse").reset_index(drop=True)

print("Best Hyperparameters:", best_params)
print("Best Validation MSE:", best_mse)
print("Model:", model2)

Best Hyperparameters: {'max_depth': np.int64(3), 'learning_rate': np.float64(0.05), 'n_estimators': np.int64(300), 'subsample': np.float64(0.8), 'colsample_bytree': np.float64(0.8), 'min_child_weight': np.int64(1), 'gamma': np.float64(0.0)}
Best Validation MSE: 1224.3647439375159
Model: XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=np.float64(0.8), device=None,
             early_stopping_rounds=None, enable_categorical=True,
             eval_metric=None, feature_types=None, feature_weights=None,
             gamma=np.float64(0.0), grow_policy=None, importance_type=None,
             interaction_constraints=None, learning_rate=np.float64(0.05),
             max_bin=None, max_cat_threshold=None, max_cat_to_onehot=None,
             max_delta_step=None, max_depth=np.int64(3), max_leaves=None,
             min_child_weight=np.int64(1), missing=nan,
             monotone_constraints=Non

### Final Test Set Evaluation

In [16]:
xgb_wage_model = model2
xgb_wage_val_mse = best_mse
xgb_wage_test_pred = xgb_wage_model.predict(X_test_encoded)
xgb_wage_test_mse = custom_mse(y_test, xgb_wage_test_pred)

print("XGBoost (Wage) — Best Hyperparameters:", best_params)
print("XGBoost (Wage) — Validation MSE:", xgb_wage_val_mse)
print("XGBoost (Wage) — Test MSE:", xgb_wage_test_mse)

XGBoost (Wage) — Best Hyperparameters: {'max_depth': np.int64(3), 'learning_rate': np.float64(0.05), 'n_estimators': np.int64(300), 'subsample': np.float64(0.8), 'colsample_bytree': np.float64(0.8), 'min_child_weight': np.int64(1), 'gamma': np.float64(0.0)}
XGBoost (Wage) — Validation MSE: 1224.3647439375159
XGBoost (Wage) — Test MSE: 1231.5370611865217


---
## Final Comparison
Validation MSE and test MSE for both models, across both datasets.

In [17]:
comparison_df = pd.DataFrame([
    {"Dataset": "RG-Credit", "Model": "Decision Tree", "Validation MSE": dt_credit_val_mse, "Test MSE": dt_credit_test_mse},
    {"Dataset": "RG-Credit", "Model": "XGBoost", "Validation MSE": xgb_credit_val_mse, "Test MSE": xgb_credit_test_mse},
    {"Dataset": "RG-Wage", "Model": "Decision Tree", "Validation MSE": dt_wage_val_mse, "Test MSE": dt_wage_test_mse},
    {"Dataset": "RG-Wage", "Model": "XGBoost", "Validation MSE": xgb_wage_val_mse, "Test MSE": xgb_wage_test_mse},
])

print("Final Comparison (Decision Tree vs. XGBoost, both datasets):")
comparison_df

Final Comparison (Decision Tree vs. XGBoost, both datasets):


,Dataset,Model,Validation MSE,Test MSE
0,RG-Credit,Decision Tree,13476.016667,24064.737500
1,RG-Credit,XGBoost,7060.304205,5181.186680
2,RG-Wage,Decision Tree,1303.521108,1298.733021
3,RG-Wage,XGBoost,1224.364744,1231.537061
